# OPSD gap on GSM8K

Measures the student vs self-teacher gap, the predictor behind Tufa Labs' OPSD law,
on math instead of code. No training.

**Runtime -> Change runtime type -> GPU (T4).**

In [1]:
!nvidia-smi

Fri Oct  2 13:53:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# repo and dependencies (~3-5 min)
REPO = "https://github.com/kaj04/opsd-gap-gsm8k.git"
!git clone -q $REPO /content/opsd-gap-gsm8k || echo 'already cloned'
%cd /content/opsd-gap-gsm8k
!pip install -q vllm datasets

fatal: could not read Username for 'https://github.com': No such device or address
already cloned
[Errno 2] No such file or directory: '/content/opsd-gap-gsm8k'
/content
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## A. What GSM8K problems look like

In [3]:
from datasets import load_dataset
import textwrap

ds = load_dataset("openai/gsm8k", "main", split="test")
print("problems in the test split:", len(ds))

for i in range(3):
    r = ds[i]
    print("=" * 78)
    print("PROBLEM %d" % i)
    print("=" * 78)
    print(textwrap.fill(r["question"], 76))
    print("\n--- reference solution (human written) ---")
    print(r["answer"])
    print()

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

problems in the test split: 1319
PROBLEM 0
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every
morning and bakes muffins for her friends every day with four. She sells the
remainder at the farmers' market daily for $2 per fresh duck egg. How much
in dollars does she make every day at the farmers' market?

--- reference solution (human written) ---
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18

PROBLEM 1
A robe takes 2 bolts of blue fiber and half that much white fiber.  How many
bolts in total does it take?

--- reference solution (human written) ---
It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3

PROBLEM 2
Josh decides to try flipping a house.  He buys a house for $80,000 and then
puts in $50,000 in repairs.  This increased the value of the house by 150%.
How much profit did he make?

--- reference solution (human wri

In [4]:
# how long the problems are and how many steps they take
import numpy as np

sub = ds.select(range(200))
words = [len(r["question"].split()) for r in sub]
steps = [r["answer"].count("<<") for r in sub]  # <<48/2=24>> marks one calculation
print("words per problem: median %d (min %d, max %d)" % (
    np.median(words), min(words), max(words)))
print("calculations per solution: median %d (min %d, max %d)" % (
    np.median(steps), min(steps), max(steps)))
print("distribution:", {k: steps.count(k) for k in sorted(set(steps))})

words per problem: median 41 (min 18, max 110)
calculations per solution: median 3 (min 0, max 7)
distribution: {0: 4, 1: 9, 2: 71, 3: 45, 4: 39, 5: 19, 6: 8, 7: 5}


## B. What the self-teacher prompts look like

Synthetic examples, no GPU. Check that no privileged context leaks the final answer.

In [14]:
!python src/inspect_prompts.py

CONDITION: none
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

Solve the problem step by step. End your reply with the final numeric answer on its own line, in the form: #### <number>

>>> leaks the final answer (72)? no

CONDITION: static_math
You are an expert at grade-school math word problems. You master:
- Translating a word problem into explicit quantities and a sequence of arithmetic steps.
- Multi-step reasoning: rates, ratios, percentages, unit conversions, totals and differences.
- Keeping track of what each intermediate number means before combining it with another.

Approach EVERY problem with this methodology:

1. Read the problem twice. List every quantity given, with its unit and meaning.
2. Identify what is being asked, and which quantities it depends on.
3. Work forward one step at a time. After each step, state what the new number means.
4. Do the arithmetic c

## C. Smoke test

In [15]:
# 10 problems, 4 samples, 3 conditions (~5 min)
!python src/run_eval.py --n-problems 10 --n-samples 4 --tag smoke \
    --conditions none,feedback_binary,gt_solution

config.json: 100% 726/726 [00:00<00:00, 3.12MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 21.0MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 115MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 127MB/s]

tokenizer.json: downloading bytes:  30% 3.40M/11.4M [00:01<00:02, 3.04MB/s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:01<00:00, 3.00MB/s,  330kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:01<00:00, 10.1MB/s, 1.11MB/s  ]
INFO 10-02 14:07:04 [api_utils.py:286] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3-1.7B'}
INFO 10-02 14:07:25 [model.py:692] Resolved architecture: Qwen3ForCausalLM
WARNING 10-02 14:07:25 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 14:07:25 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-

In [16]:
# the real prompts the model saw during the smoke test
import json

ex = json.load(open("results/smoke_prompt_examples.json", encoding="utf-8"))
for cond, items in ex.items():
    if not items:
        print("(%s: no sample got context)\n" % cond)
        continue
    print("#" * 78)
    print("# CONDITION: %s   (problem %d, sample %d)" % (
        cond, items[0]["problem"], items[0]["sample"]))
    print("#" * 78)
    print(items[0]["prompt"])
    print()

(none: no sample got context)

##############################################################################
# CONDITION: feedback_binary   (problem 2, sample 2)
##############################################################################
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Solve the problem step by step. End your reply with the final numeric answer on its own line, in the form: #### <number>

The following is feedback from your unsuccessful earlier attempt:

Your previous answer was 120000.0, which is incorrect.

Correctly solve the original question.

##############################################################################
# CONDITION: gt_solution   (problem 2, sample 2)
##############################################################################
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in

In [18]:
!git pull -q && python src/analyze.py --tag smoke

student acc = 0.675   (Qwen/Qwen3-1.7B, 10 problems x 4 samples)
condition                      acc      gap   vs none  recovery   vs none    ctx
binary feedback              0.750   +0.075    -0.050     0.308    -0.154     13
none (baseline)              0.800   +0.125    +0.000     0.462    +0.000      0
gold solution (ceiling)      0.875   +0.200    +0.075     0.692    +0.231     13

recovery = share of the student's failed rollouts the self-teacher fixes
vs none  = same number minus `none`, which is plain resampling
ctx      = failed rollouts that actually received privileged context
failed rollouts: 13

wrote: /content/opsd-gap-gsm8k/results/gaps.csv and /content/opsd-gap-gsm8k/figures/gaps.png


In [20]:
!python src/run_eval.py --n-problems 150 --n-samples 4 --tag run --conditions none,feedback_binary,feedback_diag,hints,gt_solution

INFO 10-02 14:21:29 [api_utils.py:286] non-default args: {'max_model_len': 4096, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen3-1.7B'}
INFO 10-02 14:21:30 [model.py:692] Resolved architecture: Qwen3ForCausalLM
WARNING 10-02 14:21:30 [model.py:2311] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 10-02 14:21:30 [model.py:2364] Casting torch.bfloat16 to torch.float16.
INFO 10-02 14:21:30 [model.py:2030] Using max model len 4096
INFO 10-02 14:21:30 [scheduler.py:288] Chunked prefill is enabled with max_num_batched_tokens=8192.
Parse safetensors files: 100% 2/2 [00:00<00:00,  2.15it/s]
WARNING 10-02 14:21:31 [vllm.py:1547] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 10-02 14:21:31 [vllm.py:1570] Inductor compilation was disabled by user settings, optimi

In [22]:
!git pull -q && python src/analyze.py --tag run

student acc = 0.812   (Qwen/Qwen3-1.7B, 150 problems x 4 samples)
condition                recovery   vs none    95% CI on vs none     on ctx    ctx
none (baseline)             0.221    +0.000   [+0.000, +0.000]        nan      0
binary feedback             0.301    +0.080   [-0.101, +0.176]        nan    113
diagnostic feedback         0.425    +0.204   [+0.033, +0.283]        nan    113
peer hints                  0.469    +0.248   [+0.093, +0.419]        nan     57
gold solution (ceiling)     0.779    +0.558   [+0.308, +0.628]        nan    113

recovery = share of the student's failed rollouts the self-teacher fixes
vs none  = minus `none`, which is plain resampling. CI is bootstrapped
           over problems, so it covers the difference, not the level
on ctx   = same recovery, restricted to failures that got context
ctx      = how many failed rollouts actually received context
failed rollouts: 113

wrote: /content/opsd-gap-gsm8k/results/gaps.csv and /content/opsd-gap-gsm8k/figure

## D. Full run and analysis

In [ ]:
# all conditions (slow - start with 100 problems)
!python src/run_eval.py --n-problems 100 --n-samples 8 --tag run

In [ ]:
# prompts from the full run, one condition at a time
import json

ex = json.load(open("results/run_prompt_examples.json", encoding="utf-8"))
COND = "feedback_diag"  # change to inspect the others
for item in ex[COND]:
    print("-" * 78)
    print(item["prompt"])
    print()

In [ ]:
# table and plot
!python src/analyze.py --tag run
from IPython.display import Image
Image('figures/gaps.png')

In [ ]:
# download the results to commit them back into the repo
from google.colab import files
!zip -qr results.zip results figures
files.download('results.zip')